In [1]:
%load_ext bigquery_magics

BigQueryは、一般的なRDBMSでは扱いにくい`ネストされたデータ（Nested Data）`をサポートしている。

主なデータ型は次のとおりである。

- `ARRAY`：配列
- `STRUCT`：オブジェクト・レコード
- `ARRAY<STRUCT>`：オブジェクトの配列

実務では、単純な配列よりも`ARRAY<STRUCT>`形式がよく使用される。(Pythonの [{}, {}, ...] 型)

## 1. ARRAY

一つの列に複数の値を配列として保存できる。

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        ['読書', '東京散策', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobbies,
    name
FROM tmp;

| 行 | hobby | name |
|---:|---|---|
| 1 | サッカー<br>映画<br>ゲーム | 千尋 |

#### # 特定の要素を取得

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        ['読書', '東京散策', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobbies[0] AS first_hobby, -- hobby[0]は hobby[OFFSET(0)] の形に変更されて実行
    name
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,first_hobby,name
0,読書,千尋


#### # 数値 ARRAY(配列)

In [7]:
%%bigquery

WITH tmp AS (
    SELECT [10, 20, 30, 40] AS scores
)

SELECT scores
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,scores
0,"[10, 20, 30, 40]"


## 2. ARRAY型でよく使用する機能

#### 1）UNNESTで配列を複数行に展開する

`UNNEST`は、配列を受け取り、各要素を複数の行に展開してテーブルとして返す。

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        ['サッカー', '映画', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobby,
    name
FROM tmp
CROSS JOIN UNNEST(hobbies) AS hobby;

- `hobbies`配列の各要素を`hobby`という行に分離する。
- `tmp`と`UNNEST(hobbies)`を`CROSS JOIN`するため、3行が返される。

Query is running:   0%|          |

Downloading:   0%|          |

,hobby,name
0,サッカー,千尋
1,映画,千尋
2,ゲーム,千尋


### # カンマを使用した省略形

In [10]:
%%bigquery

WITH tmp AS (
    SELECT
        ['サッカー', '映画', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobby,
    name
FROM tmp, UNNEST(hobbies) AS hobby;

Query is running:   0%|          |

Downloading:   0%|          |

,hobby,name
0,サッカー,千尋
1,映画,千尋
2,ゲーム,千尋


次の二つは同じ意味である。

```sql
FROM tmp
CROSS JOIN UNNEST(hobbies) AS hobby;
```

```sql
FROM tmp, UNNEST(hobbies) AS hobby;
```

> BigQueryはカンマを使用した結合もサポートするが、実務では可読性の高いANSI JOIN構文(JOIN ... ON )を使用することが望ましい。

#### # カンマ結合とANSI JOINの比較

次の二つのクエリは同じ意味である。

```sql
SELECT *
FROM a, b
WHERE a.id = b.id;

SELECT *
FROM a
INNER JOIN b
    ON a.id = b.id;
```

次の二つも同じ意味である。

```sql
SELECT *
FROM a, b;

SELECT *
FROM a
CROSS JOIN b;
```